# 🏰 VGG16 — Notes + Interview
---
> **Simple English** | **Interview Ready** | Year: 2014 | Creator: Oxford VGG Group

## 📌 What is VGG16? (Simple English)
- VGG = **Visual Geometry Group** (Oxford researchers)
- Key insight: **depth matters** — stacking many 3×3 conv layers is better than few large ones
- VGG16 = **16 weight layers** (13 conv + 3 dense)
- Very simple, uniform design — just 3×3 convolutions everywhere
- Popular for **transfer learning** — great pretrained features
- Limitation: **very large** (138M params, 500MB+)

## 🔑 VGG16 Architecture
```
Input (224×224×3)
→ [Conv(64)×2] → MaxPool        # 112×112×64
→ [Conv(128)×2] → MaxPool       # 56×56×128
→ [Conv(256)×3] → MaxPool       # 28×28×256
→ [Conv(512)×3] → MaxPool       # 14×14×512
→ [Conv(512)×3] → MaxPool       # 7×7×512
→ Flatten
→ Dense(4096) → Dense(4096) → Dense(1000,softmax)
```

## 🧱 Why 3×3 filters?
- Two 3×3 layers = receptive field of one 5×5 but fewer params (2×9=18 vs 25)
- Three 3×3 layers = receptive field of one 7×7 but much fewer params (3×9=27 vs 49)
- More non-linearities = better feature learning!

In [ ]:
import tensorflow as tf

# ── VGG16 from scratch ──
def build_vgg16(input_shape=(224,224,3), num_classes=1000):
    model = tf.keras.Sequential([
        # Block 1
        tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same',input_shape=input_shape),
        tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),

        # Block 2
        tf.keras.layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),

        # Block 3
        tf.keras.layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(256,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),

        # Block 4
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),

        # Block 5
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.Conv2D(512,(3,3),activation='relu',padding='same'),
        tf.keras.layers.MaxPooling2D(2,2),

        # Classifier
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(4096,activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(4096,activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes,activation='softmax'),
    ], name='VGG16')
    return model

vgg = build_vgg16()
vgg.summary()
print(f"\nTotal params: {vgg.count_params():,}  (~138M — very large!)")

In [ ]:
# Pretrained VGG16 (Transfer Learning)
vgg16_pretrained = tf.keras.applications.VGG16(
    weights='imagenet',    # load ImageNet pretrained weights
    include_top=False,     # exclude the Dense classifier head
    input_shape=(224,224,3)
)
vgg16_pretrained.summary()

# Use as feature extractor — freeze all layers
vgg16_pretrained.trainable = False

# Add custom head for new task
x = vgg16_pretrained.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(10, activation='softmax')(x)

model_transfer = tf.keras.Model(vgg16_pretrained.input, output)
print(f"\nTrainable params: {sum([tf.size(w).numpy() for w in model_transfer.trainable_weights]):,}")
print("Only the new Dense head is trainable — VGG16 frozen ✅")

## 🗣️ Interview Q&A

**Q: What is VGG16?**
> A 16-layer CNN by Oxford's VGG group. Uses only 3×3 convolutions throughout. Won ImageNet 2014 (localization). Known for simplicity and strong transfer learning features.

**Q: Why does VGG use only 3×3 filters?**
> Three 3×3 layers have same receptive field as one 7×7 but (a) fewer parameters (27 vs 49 per channel), (b) two extra non-linearities (ReLUs), making it more powerful.

**Q: What is the biggest disadvantage of VGG?**
> Size! 138M parameters, ~500MB. The three Dense(4096) layers have most params. Very slow to train and deploy. ResNet with skip connections is far more efficient.

**Q: How is VGG16 used for transfer learning?**
> Load pretrained VGG16 (without top Dense layers), freeze weights, add new Dense layers for your specific classes. The conv layers already know edges/textures/shapes from ImageNet.